In [2]:
%load_ext autoreload
%autoreload 2
import os,sys
import torch
import numpy as np
import pandas as pd
import scanpy as sc
import PINN
from PINN import reader, models
from torchdiffeq import odeint
from TorchDiffEqPack.odesolver import odesolve
from torchdyn.core import NeuralODE
import matplotlib.pyplot as plt
from matplotlib import cm

os.chdir("/ssd/users/Wergillius/Project/PINN_dynamics")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:

device = 'cuda:0'

adata = sc.read_h5ad(f"data/tom_pos.h5ad")
cellstate_key = "DM_scaled"
n_dimension = 10
base_cellstate = adata.obsm[cellstate_key][:,:n_dimension].copy()
Dataset = reader.TwoTimpepoint_AnnDS(
                    AnnData=adata, 
                    timepoint_idx=9, 
                    n_dimension=10,
                    cellstate_key=cellstate_key,  #'DM_EigenVector'
                    log_transform=False,
                    norm_time=False,
                    deltax_key=None,
                    batchsize= 100)

Computing density :
`density_funs` not specified, use gaussian kde


In [4]:
model_ckpt = "logs/tom_pos-DM_scaled_n9/pde_params_tsense/lightning_logs/version_0/checkpoints/epoch=257-total_loss=0.76506275.ckpt"
model_ckpt = "logs/tom_pos-DM_scaled_n[0, 1, 2, 3, 4, 6, 8]/pde_params_tsense/lightning_logs/version_4/checkpoints/epoch=87-total_loss=1.34044445.ckpt"
pde_model = models.pde_params.load_from_checkpoint(model_ckpt).to(device)

/ssd/users/Wergillius/miniforge3/envs/PINN_torch/lib/python3.9/site-packages/pytorch_lightning/utilities/migration/utils.py:49: PossibleUserWarning: The loaded checkpoint was produced with Lightning v2.3.0, which is newer than your current Lightning version: v1.9.5
  rank_zero_warn(


In [6]:
from tqdm import tqdm

In [7]:
# FORWARD 
u_pred_ls = []
v_ls = []
D_ls = []
g_ls = []
chunk_size= 500

s_ts = Dataset.s.float().to(device)
t_ts = Dataset.t_b.float().to(device)

with torch.no_grad():
    for i in tqdm(range(0, len(t_ts), chunk_size)):                              
        s_in = s_ts[i:i+chunk_size]
        t_in = t_ts[i:i+chunk_size]

        v_pred = pde_model.v(s_in, t_in)
        g_pred = pde_model.g(s_in, t_in)
        D_pred = pde_model.D(s_in, t_in)
        
        v_ls.append(v_pred.detach().cpu().numpy())
        g_ls.append(g_pred.detach().cpu().numpy())
        D_ls.append(D_pred.detach().cpu().numpy())

        if "u" in dir(pde_model):
            u_pred = torch.exp(pde_model.u(s_in, t_in))
            u_pred_ls.append(u_pred.detach().cpu().numpy())


 22%|██▏       | 198/890 [00:00<00:00, 1038.76it/s]

100%|██████████| 890/890 [00:00<00:00, 1191.77it/s]


In [8]:
adata

AnnData object with n_obs × n_vars = 49390 × 4814
    obs: 'n_genes', 'n_counts', 'mt_count', 'mt_frac', 'doublet_scores', 'predicted_doublets', 'xist_logn', 'Ygene_logn', 'xist_bin', 'Ygene_bin', 'sex_adata', 'biosample_id', 'cellid', 'RBG', 'SLXid', 'index', '10xsample_description', 'sex_mixed', 'sex_meta', 'mouse_id', 'sortedcells', 'expected_cells_10x', 'cellranger_cellsfound', 'chemistry', 'tom', 'expdate', 'batch', 'timepoint_tx_days', 'start_age', 'sample_id', 'countfile', 'S_score', 'G2M_score', 'phase', 'leiden', 'SLX', 'plate_sorted', 'plate_rearranged', 'well_sorted', 'well_rearranged', 'set_index', 'CI_index', 'mouse_platelabel', 'sort_method', 'sample.name', 'population', 'sex', 'countfolder', 'batch_plate_sorted', 'data_type', 'sex_combined', 'longname', 'anno_man', 'leiden_DM', 'HSCscore', 'nn_HSCscore', 'isroot', 'dpt_pseudotime', 'leiden_orig', 'logk', 'net_prolif', 'log10SR', 'log_density_at_E3', 'log_density_at_E7', 'log_density_at_E12', 'log_density_at_E12_clip', 'l

# clu10

In [9]:
hspc_dir = "/ssd/users/Wergillius/Project/PINN_dynamics/data/HSCP_PD_params/clu10"
clu10_df = pd.read_csv(f"{hspc_dir}/input_pseudo_dyn_rolling_clu_10_dpt.csv")

In [11]:
np.intersect1d(clu10_df['sample'].values, adata.obs_names)

array([], dtype=object)